# 🧹 Germany Housing Market — Phase 2: Clean the Data

## 🎯 Goal of this phase
Turn the **raw** file (`data/raw/immo_data.csv`) into a **clean** dataset we can analyze. We will remove duplicates, drop impossible values, handle missing data, and create useful new columns — then save the result to `data/processed/immo_data_clean.csv`.

## 📖 How to use this notebook
Same rules as Phase 1: read the Markdown, then **write the code yourself** in each cell (replacing the `# TODO`), run it, and compare with the **Expected output** note.

> 💡 Phase 1 taught you what is wrong with the data. This phase fixes it. If you did not finish Phase 1, go back and do it first — your findings will tell you what to clean.

## 📥 Carried over from Phase 1 (your open items)

Phase 1 is closed as *exploration* — nothing below has been acted on yet. These findings are its output and the **input** to the cleaning below. Every number was verified against `data/raw/immo_data.csv` on 2026-09-17.

### A. Decisions this notebook has to make

| Phase 1 finding | Where it gets decided |
|---|---|
| `regio1` == `geo_bln` (identical, 16 states, 0 nulls) | drop one of the pair → **Step 4** |
| `regio2` == `geo_krs` (identical, 419 districts, 0 nulls) | drop one of the pair → **Step 4** |
| `regio2` is a **district**, not a city | build `city` from **`regio3`** instead → **Step 5** |
| `date` holds only **4** snapshots: `Sep18`, `May19`, `Oct19`, `Feb20`, unequal gaps, stops Feb20 | parse in **Step 5**; the caveat must survive into Phase 3 |
| 7 columns are > 60 % empty | drop list → **Step 4** |
| `totalRent` is 15.07 % empty (≈ 40,500 rows) vs `baseRent` 0 % | decide: fill, drop, or analyse `baseRent` as the primary price → **Step 4 / 7** |

### B. Verified extremes — use these as your Step 3 bounds

| Column | min | max | note |
|---|---|---|---|
| `baseRent` | 0 | 9,999,999 | std **19,536** vs mean **694** → a handful of rows would dominate every average |
| `totalRent` | 0 | 15,751,535 | 15.07 % missing |
| `livingSpace` | 0 | 111,111 | m² |
| `noRooms` | 1 | 999.99 | |

### C. Verified list of columns above 60 % missing (Step 4 drop list)

`telekomHybridUploadSpeed` (83.25 %), `electricityBasePrice` (82.58 %), `electricityKwhPrice` (82.58 %), `energyEfficiencyClass` (71.07 %), `lastRefurbish` (69.98 %), `heatingCosts` (68.19 %), `noParkSpaces` (65.39 %).

### D. Text style (Step 5)

Names use `_` instead of spaces and drop umlauts: `Nordrhein_Westfalen`, `Äußere_Neustadt_Antonstadt`, `Südliche_Weinstraße_Kreis`. Tableau will show these verbatim until you clean them.

### E. Stays behind in Notebook 1

- the clean `Run All` state — notebook 1 now executes 40 → 48 in order. Keep it that way: a phase is only "done" when it reproduces.

> 🚨 **Scope risk to carry forward:** with only 4 irregular time points, plan questions 6, 8 and 9 ("rising fastest", "price appreciation", "ROI trend") cannot be answered credibly. Decide in Phase 3 whether to reframe them as *snapshot comparisons* instead — for example "how did rent per m² in the top 10 cities move between Sep18 and Feb20" rather than a continuous trend.

## Step 1 — Setup: imports + reload the raw data

**What & why:** Notebooks do not remember variables from other notebooks, so we start fresh: import the tools, then load the raw CSV again into a new `df`.

**Input:** `../data/raw/immo_data.csv`.

**Hints (not the answer):**
- Import `pandas as pd` and `numpy as np` (as in Phase 1).
- Read the CSV with `pd.read_csv(...)` into `df`, then print `df.shape`.
- Keep the raw data untouched — we only read it. All cleaning happens on a copy we build step by step.
- Set the float display format as in Phase 1 (`pd.set_option(...)`) so large numbers stay readable.

**Expected output (verified):** exactly `(268850, 49)` — the same shape Phase 1 saw. If your column count differs, stop and investigate *before* cleaning anything.

In [ ]:
# TODO 1 — import pandas and numpy, load the raw CSV into df, print df.shape

## Step 2 — Remove duplicate listings

**What & why:** The same apartment may appear more than once. Each listing has a unique id column named `scoutId` — rows with the same `scoutId` are duplicates.

**Input:** `df`.

**Hints (not the answer):**
- Count duplicates first: `df.duplicated(subset=['scoutId']).sum()`.
- Then remove them with `df.drop_duplicates(subset=['scoutId'], keep='first')`.
- Save the result back into `df` and print the number of rows **before vs after**.

**Expected output:** a small duplicate count, then a new row count that is lower.

In [ ]:
# TODO 2 — count duplicates on scoutId, drop them, report rows before vs after

## Step 3 — Remove impossible values

**What & why:** Some rows are clearly data errors (a 1 m² flat, a €0 rent). Your Phase 1 numbers show how extreme they are: `baseRent` runs from 0 to 9,999,999 with a standard deviation of 19,536 against a mean of 694 — meaning a handful of rows would otherwise dominate every average you compute later. Removing them is not optional.

**Input:** `df` (see the carried-over table in section B above).

**Hints (not the answer):**
- Build **boolean masks** (e.g. `df['baseRent'] >= 10`) and combine them with `&`. Remember: `&` needs each comparison in parentheses.
- Suggested starting bounds (adjust to your own Phase 1 findings): `baseRent` 10–10,000, `livingSpace` 10–500, `noRooms` 1–20.
- Keep the rows that pass **all** checks, then report how many rows were removed.
- 💡 Alternative worth knowing: **quantile clipping** — `df['baseRent'].quantile(0.99)` lets the data define "extreme" instead of you hard-coding a limit. Try one approach first; you can compare survivors later.

**Expected output:** a message like `Removed 8,000 impossible rows (260,850 remaining)` — your numbers will differ. What matters is that you can *justify* each bound, not that you match a target.

In [ ]:
# TODO 3 — filter out impossible values using boolean masks; report rows removed

## Step 4 — Handle missing values

**What & why:** Empty cells (`NaN`) remain. Strategy: (1) **drop** columns that are mostly empty (>60 %), and (2) **fill** the rest sensibly — numbers with the median, text with `'unknown'` or the most common value.

**Input:** `df`.

**Hints (not the answer):**
- Recompute missing % with `df.isna().mean()` and find columns above 0.60.
- Drop those columns with `df.drop(columns=[...])`.
- For the rest: use `.fillna(...)` — the **median** for numeric columns (e.g. `yearConstructed`), `'unknown'` for text columns.

**Expected output:** a message listing dropped columns, and after filling, a far lower total missing count.

In [ ]:
# TODO 4 — drop columns with >60% missing; fill numeric with median and text with 'unknown'

## Step 5 — Fix dates and location names

**What & why:** (a) The `date` column is text like `Feb20` — it should become a real date. (b) Location names use `_` instead of spaces (e.g. `Nordrhein_Westfalen`), so we build readable `state` and `city` columns.

**Input:** `df`.

**⚠️ Corrected by Phase 1:** `regio2` is **not** a city — it is the district (Kreis) and an exact duplicate of `geo_krs`. The city / neighbourhood column is **`regio3`** (~8,700 unique values, e.g. `Schüren`, `Äußere_Neustadt_Antonstadt`).

**Hints (not the answer):**
- Parse dates with `pd.to_datetime(df['date'], format='%b%y')` — look up what `%b` and `%y` mean. The values are English month abbreviations, which is exactly what `%b` expects.
- You should end up with exactly **4** distinct dates: Sep 2018, May 2019, Oct 2019, Feb 2020. If you get more, something went wrong.
- Clean text with `.str.replace('_', ' ')` and `.str.strip()`.
- Build `state` from `regio1` and `city` from `regio3`. Drop the redundant twins (`geo_bln`, and `geo_krs` *or* `regio2`) while you are here.
- 💡 Note: `regio3` is very granular — most places have only a few dozen listings. Decide later whether to keep it as-is, or additionally group to the district level for analysis.

**Expected output:** `date` becomes datetime with 4 unique values; new `state` and `city` columns show readable names with spaces instead of underscores.

In [ ]:
# TODO 5 — convert the date column; create clean state (from regio1) and city (from regio2) columns

## Step 6 — Feature engineering

**What & why:** Create the columns that make analysis easy: **price per m²** (rent ÷ area — the standard way to compare flats) and **size/room categories** so we can group flats later.

**Input:** `df`.

**Hints (not the answer):**
- `df['price_per_m2'] = df['baseRent'] / df['livingSpace']` — round it, and maybe clip to sane bounds (e.g. 5–100).
- Use `pd.cut(df['livingSpace'], bins=[...], labels=[...])` to make categories like `small`/`medium`/`large`. Pick your own bins that make sense.
- Do the same for rooms, e.g. a `room_category`.

**Expected output:** new columns appear; `df.shape` shows more columns than before.

In [ ]:
# TODO 6 — create price_per_m2, size_category and room_category

## Step 7 — Final cleanup and save

**What & why:** Drop any remaining rows missing critical info (`baseRent`, `livingSpace`, `noRooms`, `state`), reset the index, and save the clean file where Phase 3 will read it.

**Input:** `df`.

**Hints (not the answer):**
- `df.dropna(subset=[...])` removes rows with empty critical cells.
- `df.reset_index(drop=True)` gives a clean 0…N index.
- Save with `df.to_csv('../data/processed/immo_data_clean.csv', index=False)`.
- Print final shape and how much of the original data survived (row-loss %).

**Expected output:** a saved file, a final shape (e.g. ~250k rows) and a row-loss message.

In [ ]:
# TODO 7 — drop rows missing critical columns, reset index, save clean CSV, report final shape

## ✍️ Step 8 — Check your work

Edit this cell and answer honestly:

- [ ] Final row count ≈ what I expected? (write it: ______)
- [ ] `data/processed/immo_data_clean.csv` exists in the file explorer?
- [ ] No obvious `NaN` left in the columns we need?
- [ ] I can explain, in one sentence, what `price_per_m2` means? (write it: ______)

**Anything that confused you (notes for me):**

- 